# Franka Panda Push + SAC/HER + layered reward + MuJoCo 3

Based on `zichunxx/panda_mujoco_gym`, updated for a modern Colab stack.

## Environment stack
- MuJoCo **3.12.0**
- Gymnasium **1.3.0**
- Gymnasium-Robotics **1.4.2**
- Stable-Baselines3 **2.9.0**
- SB3-Contrib **2.9.0**

The original repository pinned MuJoCo 2.3.3 in 2023. This notebook deliberately does **not** install the repository's old `requirements.txt`; it clones only the source/assets and installs a MuJoCo 3 stack explicitly.

## Final reward

\[
r(d)=
\begin{cases}
-1-a[\tanh(kd)-\tanh(kd_s)], & d>d_s\\
-1, & d_g<d\le d_s\\
0, & d\le d_g
\end{cases}
\]

with \(d=\|\text{achieved_goal}-\text{desired_goal}\|_2\).

Far away, continuous shaping gives direction. Near the goal, the reward becomes the classical HER sparse `-1/0` form. At \(d=d_s\), both branches equal `-1`, so there is no reward jump.

## 1. Rebuild the Colab environment from scratch

This cell first removes potentially conflicting MuJoCo/Gymnasium/SB3 packages, then installs a pinned modern stack.

The original repository is cloned only for its Franka environment source and XML/assets.

**Do not run the repository's `pip install -r requirements.txt` afterward**, because that file pins MuJoCo 2.3.3.

In [ ]:
# ---------- System libraries ----------
!apt-get update -qq
!apt-get install -y -qq \
    libgl1 libglfw3 libglew2.2 libosmesa6 libosmesa6-dev ffmpeg > /dev/null

# ---------- Remove conflicting RL packages ----------
!pip uninstall -y \
    mujoco gymnasium gymnasium-robotics stable-baselines3 sb3-contrib \
    > /dev/null 2>&1 || true

# ---------- Install modern compatible stack ----------
!pip install -q --no-cache-dir --upgrade \
    "mujoco==3.12.0" \
    "gymnasium==1.3.0" \
    "gymnasium-robotics==1.4.2" \
    "stable-baselines3==2.9.0" \
    "sb3-contrib==2.9.0" \
    tensorboard matplotlib pandas tqdm rich imageio imageio-ffmpeg

# ---------- Clone Franka MuJoCo environment ----------
!rm -rf /content/panda_mujoco_gym
!git clone -q https://github.com/zichunxx/panda_mujoco_gym.git /content/panda_mujoco_gym
%cd /content/panda_mujoco_gym

print("Environment installation finished.")
print("MuJoCo 3.12.0 / Gymnasium 1.3.0 / Gymnasium-Robotics 1.4.2")
print("Stable-Baselines3 2.9.0 / SB3-Contrib 2.9.0")
print("If any of these packages were imported earlier in this runtime, restart once before continuing.")

### If Colab reports package-version conflicts

Restart the runtime once and rerun the main install cell. Do not install the repository's original requirements afterward.

In [ ]:
# No legacy fallback is used.
# Keep MuJoCo >= 3 and rerun the main installation cell after a runtime restart if needed.

In [ ]:
# ---------- Environment diagnostic ----------
import sys, os, importlib

expected = {
    "mujoco": "3.12.0",
    "gymnasium": "1.3.0",
    "gymnasium_robotics": "1.4.2",
    "stable_baselines3": "2.9.0",
    "sb3_contrib": "2.9.0",
}

print("Python:", sys.version)
print("Working directory:", os.getcwd())

actual = {}
for name, expected_version in expected.items():
    module = importlib.import_module(name)
    version = getattr(module, "__version__", "version not exposed")
    actual[name] = version
    print(f"{name}: {version} (expected {expected_version})")

import mujoco
assert int(mujoco.__version__.split(".")[0]) >= 3
assert actual["gymnasium"] == "1.3.0"
assert actual["stable_baselines3"] == "2.9.0"
assert actual["sb3_contrib"] == "2.9.0"

import panda_mujoco_gym
import gymnasium as gym

_probe = gym.make("FrankaPushSparse-v0")
_obs, _info = _probe.reset(seed=0)
print("FrankaPushSparse-v0 created successfully.")

for _ in range(3):
    _obs, _r, _terminated, _truncated, _info = _probe.step(_probe.action_space.sample())

_probe.close()
print("Environment check PASSED.")

## 2. Imports and experiment configuration

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import torch

import panda_mujoco_gym
from panda_mujoco_gym.envs.push import FrankaPushEnv

from stable_baselines3 import SAC, DDPG, HerReplayBuffer
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.env_checker import check_env
from sb3_contrib import TQC

SEED = 42

# Choose: "SAC", "DDPG", or "TQC"
ALGO = "SAC"

# Hybrid reward parameters
REWARD_A = 1.0
REWARD_K = 5.0
D_SWITCH = 0.12      # 12 cm: dense -> sparse
D_SUCCESS = 0.05     # 5 cm: success

MAX_EPISODE_STEPS = 50

# Classical -1/0 HER-style reward: terminate when the goal is reached.
TERMINATE_ON_SUCCESS = True

# 20k–50k: smoke test; 200k+: meaningful curves; 500k: benchmark-style Push run.
TOTAL_TIMESTEPS = 500_000

EVAL_FREQ = 5_000
N_EVAL_EPISODES = 20

RUN_NAME = f"{ALGO}_HER_Push_MuJoCo3_ds{D_SWITCH}_dg{D_SUCCESS}_seed{SEED}"

ROOT = Path("/content/franka_push_tanh")
TB_DIR = ROOT / "tensorboard"
MODEL_DIR = ROOT / "models"
EVAL_DIR = ROOT / "eval"
FIG_DIR = ROOT / "figures"

for p in [TB_DIR, MODEL_DIR, EVAL_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Run:", RUN_NAME)
print("Device available:", "cuda" if torch.cuda.is_available() else "cpu")

## 3. Custom Push environment with layered reward

\[
r(d)=
\begin{cases}
-1-a[\tanh(kd)-\tanh(kd_s)], & d>d_s\\
-1, & d_g<d\le d_s\\
0, & d\le d_g
\end{cases}
\]

This preserves continuous guidance far from the target and the classical HER sparse reward near the target. `compute_reward()` depends only on achieved/desired goals and is vectorized, so HER can safely recompute rewards after goal relabeling.

In [ ]:
class FrankaPushLayeredRewardEnv(FrankaPushEnv):
    def __init__(
        self,
        reward_a=1.0,
        reward_k=5.0,
        d_switch=0.12,
        d_success=0.05,
        max_episode_steps=50,
        terminate_on_success=True,
        render_mode=None,
    ):
        super().__init__(reward_type="dense", render_mode=render_mode)

        if not (0.0 < d_success < d_switch):
            raise ValueError("Require 0 < d_success < d_switch.")

        self.reward_a = float(reward_a)
        self.reward_k = float(reward_k)
        self.d_switch = float(d_switch)
        self.d_success = float(d_success)
        self.distance_threshold = self.d_success

        self.custom_max_episode_steps = int(max_episode_steps)
        self.terminate_on_success = bool(terminate_on_success)
        self._custom_elapsed_steps = 0

    def compute_reward(self, achieved_goal, desired_goal, info):
        # Vectorized for HER batches.
        achieved_goal = np.asarray(achieved_goal, dtype=np.float32)
        desired_goal = np.asarray(desired_goal, dtype=np.float32)

        d = np.linalg.norm(
            achieved_goal - desired_goal,
            axis=-1,
        )

        # Continuous outer shaping. It equals -1 at d == d_switch.
        outer_reward = (
            -1.0
            - self.reward_a
            * (
                np.tanh(self.reward_k * d)
                - np.tanh(self.reward_k * self.d_switch)
            )
        )

        # Classical sparse HER reward near the goal.
        reward = np.where(
            d > self.d_switch,
            outer_reward,
            -1.0,
        )

        # Success: penalty disappears.
        reward = np.where(
            d <= self.d_success,
            0.0,
            reward,
        )

        return np.asarray(reward, dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        self._custom_elapsed_steps = 0
        return super().reset(seed=seed, options=options)

    def step(self, action):
        obs, reward, terminated, truncated, info = super().step(action)
        self._custom_elapsed_steps += 1

        distance = float(
            np.linalg.norm(obs["achieved_goal"] - obs["desired_goal"])
        )
        success = bool(distance <= self.d_success)

        info = dict(info)
        info["is_success"] = success
        info["distance_to_goal"] = distance
        info["reward_layered"] = float(reward)
        info["reward_region"] = (
            "success"
            if success
            else "sparse_inner"
            if distance <= self.d_switch
            else "dense_outer"
        )

        terminated = success if self.terminate_on_success else False
        truncated = bool(
            self._custom_elapsed_steps >= self.custom_max_episode_steps
        )

        return obs, float(reward), bool(terminated), bool(truncated), info


class GoalMonitor(Monitor):
    def compute_reward(self, achieved_goal, desired_goal, info):
        return self.env.compute_reward(achieved_goal, desired_goal, info)

## 4. Environment factory and sanity checks

In [ ]:
def make_env(seed=SEED, render_mode=None, monitor_file=None):
    env = FrankaPushLayeredRewardEnv(
        reward_a=REWARD_A,
        reward_k=REWARD_K,
        d_switch=D_SWITCH,
        d_success=D_SUCCESS,
        max_episode_steps=MAX_EPISODE_STEPS,
        terminate_on_success=TERMINATE_ON_SUCCESS,
        render_mode=render_mode,
    )
    env.reset(seed=seed)

    if monitor_file is not None:
        env = GoalMonitor(env, filename=str(monitor_file))
    else:
        env = GoalMonitor(env)

    return env


test_env = FrankaPushLayeredRewardEnv(
    reward_a=REWARD_A,
    reward_k=REWARD_K,
    max_episode_steps=MAX_EPISODE_STEPS,
    terminate_on_success=TERMINATE_ON_SUCCESS,
)
check_env(test_env, warn=True)

obs, info = test_env.reset(seed=SEED)
print("Observation keys:", obs.keys())
print("observation shape:", obs["observation"].shape)
print("achieved_goal shape:", obs["achieved_goal"].shape)
print("desired_goal shape:", obs["desired_goal"].shape)
print("Action space:", test_env.action_space)

r_scalar = test_env.compute_reward(obs["achieved_goal"], obs["desired_goal"], {})
print("Scalar reward:", r_scalar)

ag_batch = np.stack([obs["achieved_goal"], obs["achieved_goal"]])
dg_batch = np.stack([obs["desired_goal"], obs["desired_goal"]])
r_batch = test_env.compute_reward(ag_batch, dg_batch, np.array([{}, {}]))
print("Vectorized reward shape:", r_batch.shape, "values:", r_batch)

test_env.close()

## 5. Visualize the final layered reward

Outside `D_SWITCH`, reward changes continuously with distance. Between `D_SUCCESS` and `D_SWITCH`, reward is exactly `-1`. Inside `D_SUCCESS`, reward is `0`.

In [ ]:
d = np.linspace(0.0, 0.6, 600)

outer_reward = (
    -1.0
    - REWARD_A
    * (
        np.tanh(REWARD_K * d)
        - np.tanh(REWARD_K * D_SWITCH)
    )
)

r = np.where(d > D_SWITCH, outer_reward, -1.0)
r = np.where(d <= D_SUCCESS, 0.0, r)

plt.figure(figsize=(8, 4.5))
plt.plot(d, r, label="layered reward")
plt.axvline(D_SUCCESS, linestyle="--", label=f"success = {D_SUCCESS:.2f} m")
plt.axvline(D_SWITCH, linestyle="--", label=f"switch = {D_SWITCH:.2f} m")
plt.axhline(-1.0, linewidth=1)

plt.xlabel("object-goal distance d (m)")
plt.ylabel("reward")
plt.title("Dense outer shaping + classical sparse HER inner region")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "layered_reward_function.png", dpi=180)
plt.show()

## 6. Deterministic evaluation callback

Every `EVAL_FREQ` steps this logs success rate, mean episode return, and final object-goal distance.

In [ ]:
class GoalEvalCallback(BaseCallback):
    def __init__(
        self,
        eval_env,
        eval_freq=5_000,
        n_eval_episodes=20,
        csv_path=None,
        verbose=1,
    ):
        super().__init__(verbose=verbose)
        self.eval_env = eval_env
        self.eval_freq = int(eval_freq)
        self.n_eval_episodes = int(n_eval_episodes)
        self.csv_path = Path(csv_path) if csv_path is not None else None
        self.last_eval_step = 0
        self.history = {
            "timesteps": [],
            "success_rate": [],
            "mean_reward": [],
            "mean_final_distance": [],
        }

    def _evaluate(self):
        episode_returns = []
        successes = []
        final_distances = []

        for ep in range(self.n_eval_episodes):
            obs, info = self.eval_env.reset(seed=10_000 + ep)
            ep_return = 0.0
            ever_success = False
            last_distance = np.nan

            for _ in range(MAX_EPISODE_STEPS):
                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = self.eval_env.step(action)
                ep_return += float(reward)
                ever_success = ever_success or bool(info.get("is_success", False))
                last_distance = float(info.get("distance_to_goal", np.nan))

                if terminated or truncated:
                    break

            episode_returns.append(ep_return)
            successes.append(float(ever_success))
            final_distances.append(last_distance)

        return (
            float(np.mean(successes)),
            float(np.mean(episode_returns)),
            float(np.nanmean(final_distances)),
        )

    def _on_step(self) -> bool:
        if self.num_timesteps - self.last_eval_step < self.eval_freq:
            return True

        self.last_eval_step = self.num_timesteps
        success_rate, mean_reward, mean_final_distance = self._evaluate()

        self.history["timesteps"].append(self.num_timesteps)
        self.history["success_rate"].append(success_rate)
        self.history["mean_reward"].append(mean_reward)
        self.history["mean_final_distance"].append(mean_final_distance)

        self.logger.record("eval_custom/success_rate", success_rate)
        self.logger.record("eval_custom/mean_reward", mean_reward)
        self.logger.record("eval_custom/mean_final_distance", mean_final_distance)
        self.logger.dump(self.num_timesteps)

        if self.csv_path is not None:
            pd.DataFrame(self.history).to_csv(self.csv_path, index=False)

        if self.verbose:
            print(
                f"[Eval @ {self.num_timesteps:>7d}] "
                f"success={success_rate:.3f}, "
                f"return={mean_reward:.3f}, "
                f"final_d={mean_final_distance:.4f} m"
            )
        return True

## 7. Build SAC / DDPG / TQC with HER

Default Push settings:

- MLP `[256, 256, 256]`
- batch size `512`
- buffer `1e6`
- learning rate `1e-3`
- gamma `0.95`
- tau `0.05`
- HER strategy `future`, 4 relabeled goals
- DDPG Gaussian action noise `sigma=0.2`

In [ ]:
train_env = make_env(
    seed=SEED,
    monitor_file=ROOT / "train_monitor.csv",
)

eval_env = make_env(
    seed=SEED + 1000,
    monitor_file=ROOT / "eval_monitor.csv",
)

common_kwargs = dict(
    policy="MultiInputPolicy",
    env=train_env,
    replay_buffer_class=HerReplayBuffer,
    replay_buffer_kwargs=dict(
        n_sampled_goal=4,
        goal_selection_strategy="future",
        copy_info_dict=False,
    ),
    learning_rate=1e-3,
    buffer_size=1_000_000,
    learning_starts=2_000,
    batch_size=512,
    tau=0.05,
    gamma=0.95,
    train_freq=1,
    gradient_steps=1,
    policy_kwargs=dict(net_arch=[256, 256, 256]),
    tensorboard_log=str(TB_DIR),
    verbose=1,
    seed=SEED,
    device="auto",
)

if ALGO.upper() == "SAC":
    model = SAC(
        **common_kwargs,
        ent_coef="auto",
    )

elif ALGO.upper() == "DDPG":
    n_actions = train_env.action_space.shape[-1]
    action_noise = NormalActionNoise(
        mean=np.zeros(n_actions),
        sigma=0.2 * np.ones(n_actions),
    )
    model = DDPG(
        **common_kwargs,
        action_noise=action_noise,
    )

elif ALGO.upper() == "TQC":
    tqc_kwargs = dict(common_kwargs)
    tqc_kwargs["policy_kwargs"] = dict(
        net_arch=[256, 256, 256],
        n_critics=2,
        n_quantiles=25,
    )
    model = TQC(
        **tqc_kwargs,
        ent_coef="auto",
        top_quantiles_to_drop_per_net=2,
    )

else:
    raise ValueError("ALGO must be 'SAC', 'DDPG', or 'TQC'")

print(model)

## 8. Train and save

In [ ]:
eval_callback = GoalEvalCallback(
    eval_env=eval_env,
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    csv_path=EVAL_DIR / f"{RUN_NAME}_eval.csv",
    verbose=1,
)

checkpoint_callback = CheckpointCallback(
    save_freq=50_000,
    save_path=str(MODEL_DIR),
    name_prefix=RUN_NAME,
    save_replay_buffer=False,
    save_vecnormalize=False,
)

callbacks = CallbackList([eval_callback, checkpoint_callback])

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callbacks,
    tb_log_name=RUN_NAME,
    reset_num_timesteps=True,
    progress_bar=True,
)

FINAL_MODEL_PATH = MODEL_DIR / f"{RUN_NAME}_final"
model.save(str(FINAL_MODEL_PATH))
print("Saved model:", str(FINAL_MODEL_PATH) + ".zip")

## 9. TensorBoard dashboard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/franka_push_tanh/tensorboard

## 10. Load logs and create a six-panel RL training figure

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def rolling_mean(x, window=20):
    return pd.Series(x).rolling(window=window, min_periods=1).mean().to_numpy()

def find_event_file(root_dir, run_name):
    candidates = list(Path(root_dir).rglob("events.out.tfevents.*"))
    preferred = [p for p in candidates if run_name in str(p.parent)]
    return preferred[-1] if preferred else (candidates[-1] if candidates else None)

def load_tb_scalars(event_file):
    if event_file is None:
        return {}
    ea = EventAccumulator(str(event_file), size_guidance={"scalars": 0})
    ea.Reload()
    result = {}
    for tag in ea.Tags().get("scalars", []):
        events = ea.Scalars(tag)
        result[tag] = pd.DataFrame({
            "step": [e.step for e in events],
            "value": [e.value for e in events],
        })
    return result


eval_csv = EVAL_DIR / f"{RUN_NAME}_eval.csv"
eval_df = pd.read_csv(eval_csv)

monitor_candidates = list(ROOT.glob("train_monitor*.monitor.csv")) + list(ROOT.glob("train_monitor*.csv"))
monitor_df = None
for p in monitor_candidates:
    try:
        temp = pd.read_csv(p, comment="#")
        if "r" in temp.columns:
            monitor_df = temp
            break
    except Exception:
        pass

event_file = find_event_file(TB_DIR, RUN_NAME)
tb = load_tb_scalars(event_file)

print("TensorBoard event:", event_file)
print("Available scalar tags:")
for tag in sorted(tb.keys()):
    print(" ", tag)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0, 0].plot(eval_df["timesteps"], eval_df["success_rate"], marker="o", markersize=3)
axes[0, 0].set_title("Evaluation Success Rate")
axes[0, 0].set_xlabel("Environment steps")
axes[0, 0].set_ylabel("Success rate")
axes[0, 0].set_ylim(-0.02, 1.02)

axes[0, 1].plot(eval_df["timesteps"], eval_df["mean_reward"])
axes[0, 1].set_title("Evaluation Mean Return")
axes[0, 1].set_xlabel("Environment steps")
axes[0, 1].set_ylabel("Return")

axes[0, 2].plot(eval_df["timesteps"], eval_df["mean_final_distance"])
axes[0, 2].set_title("Evaluation Final Distance")
axes[0, 2].set_xlabel("Environment steps")
axes[0, 2].set_ylabel("Distance (m)")

if monitor_df is not None and "r" in monitor_df.columns:
    y = monitor_df["r"].to_numpy()
    axes[1, 0].plot(y, alpha=0.25, label="episode return")
    axes[1, 0].plot(rolling_mean(y, 20), label="rolling mean")
    axes[1, 0].legend()
axes[1, 0].set_title("Training Episode Return")
axes[1, 0].set_xlabel("Episode")
axes[1, 0].set_ylabel("Return")

if "train/actor_loss" in tb:
    d_actor = tb["train/actor_loss"]
    axes[1, 1].plot(d_actor["step"], d_actor["value"])
axes[1, 1].set_title("Actor Loss")
axes[1, 1].set_xlabel("Environment steps")
axes[1, 1].set_ylabel("Loss")

if "train/critic_loss" in tb:
    d_critic = tb["train/critic_loss"]
    axes[1, 2].plot(d_critic["step"], d_critic["value"])
axes[1, 2].set_title("Critic Loss")
axes[1, 2].set_xlabel("Environment steps")
axes[1, 2].set_ylabel("Loss")

for ax in axes.flat:
    ax.grid(alpha=0.25)

fig.suptitle(f"{ALGO} + HER | FrankaPush | layered reward", fontsize=15)
plt.tight_layout()
out = FIG_DIR / f"{RUN_NAME}_training_dashboard.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", out)

## 11. Optional SAC/TQC entropy-coefficient curve

In [ ]:
if "train/ent_coef" in tb:
    d_ent = tb["train/ent_coef"]
    plt.figure(figsize=(7, 4))
    plt.plot(d_ent["step"], d_ent["value"])
    plt.title("Entropy Coefficient")
    plt.xlabel("Environment steps")
    plt.ylabel("alpha")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{RUN_NAME}_entropy_coef.png", dpi=180)
    plt.show()
else:
    print("No entropy coefficient tag (expected for DDPG).")

## 12. Final deterministic evaluation

In [ ]:
def evaluate_final(model, n_episodes=50):
    env = make_env(seed=SEED + 5000)
    results = []

    for ep in range(n_episodes):
        obs, info = env.reset(seed=SEED + 5000 + ep)
        ep_return = 0.0
        ever_success = False
        min_distance = np.inf
        final_distance = np.nan

        for _ in range(MAX_EPISODE_STEPS):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)

            ep_return += float(reward)
            ever_success |= bool(info["is_success"])
            final_distance = float(info["distance_to_goal"])
            min_distance = min(min_distance, final_distance)

            if terminated or truncated:
                break

        results.append({
            "episode": ep,
            "return": ep_return,
            "success": float(ever_success),
            "final_distance": final_distance,
            "min_distance": min_distance,
        })

    env.close()
    return pd.DataFrame(results)

final_eval = evaluate_final(model, n_episodes=50)
display(final_eval.describe())

print("Final success rate:", final_eval["success"].mean())
print("Mean final distance:", final_eval["final_distance"].mean())

final_eval.to_csv(EVAL_DIR / f"{RUN_NAME}_final_eval.csv", index=False)

## 13. Optional: render one trained episode to MP4

Uncomment if you want a video in addition to plots.

In [ ]:
# For GPU Colab off-screen rendering, set before importing mujoco in a fresh runtime:
# %env MUJOCO_GL=egl
# For CPU-only off-screen rendering, OSMesa may be used instead:
# %env MUJOCO_GL=osmesa
# import imageio.v2 as imageio
#
# video_env = make_env(seed=SEED + 9000, render_mode="rgb_array")
# obs, info = video_env.reset(seed=SEED + 9000)
# frames = []
#
# for _ in range(MAX_EPISODE_STEPS):
#     frame = video_env.render()
#     if frame is not None:
#         frames.append(frame)
#     action, _ = model.predict(obs, deterministic=True)
#     obs, reward, terminated, truncated, info = video_env.step(action)
#     if terminated or truncated:
#         break
#
# video_env.close()
# video_path = FIG_DIR / f"{RUN_NAME}_episode.mp4"
# imageio.mimsave(video_path, frames, fps=20)
# print(video_path)

## 14. Recommended comparison

After SAC works, change only `ALGO` and rerun:

- `DDPG + HER`
- `SAC + HER`
- `TQC + HER`

Keep the same reward parameters \(a,k,d_{switch},d_{success},R_{success}\), seeds, network, HER settings, training steps and evaluation procedure. Compare success rate, convergence speed, actor/critic loss, and final distance.